# Customer Churn Prediction – Statistical Analysis

## Objective

The objective of this notebook is to statistically investigate
relationships between customer characteristics and churn.

Exploratory Data Analysis (EDA) identified several potential patterns.
Statistical hypothesis testing is used to determine whether selected
differences or associations are statistically significant.

## Statistical Methods

The analysis will use appropriate statistical tests for:

- Numerical variables vs. churn
- Categorical variables vs. churn

The interpretation will consider:

- Null hypothesis
- Alternative hypothesis
- Test statistic
- p-value
- Statistical significance
- Business interpretation

## Important

Statistical significance does not imply causation.
The tests identify evidence of association or difference, not proof
that one variable causes customer churn.

## 1. Import Libraries

The required libraries are imported for data manipulation and
statistical hypothesis testing.

In [1]:
import pandas as pd
import numpy as np

from scipy import stats

## 2. Load Cleaned Dataset

The cleaned dataset generated during the data-cleaning stage is loaded
for statistical analysis.

In [2]:
processed_path = "../data/processed/telco_customer_churn_cleaned.csv"

df = pd.read_csv(processed_path)

print("Cleaned dataset loaded successfully.")
print("Shape:", df.shape)

Cleaned dataset loaded successfully.
Shape: (7043, 21)


## 3. Welch's t-test – Monthly Charges vs Churn

We compare the mean MonthlyCharges between customers who churned and
customers who did not churn.

Welch's independent two-sample t-test is used because it does not
require the two groups to have equal variances.

In [3]:
monthly_charges_no_churn = df.loc[
    df["Churn"] == "No",
    "MonthlyCharges"
]

monthly_charges_churn = df.loc[
    df["Churn"] == "Yes",
    "MonthlyCharges"
]

In [4]:
t_stat, p_value = stats.ttest_ind(
    monthly_charges_no_churn,
    monthly_charges_churn,
    equal_var=False
)

print("t-statistic:", t_stat)
print("p-value:", p_value)

t-statistic: -18.407526676414673
p-value: 8.59244933154705e-73


In [5]:
alpha = 0.05

if p_value < alpha:
    print("Result: Reject the null hypothesis.")
    print("There is statistically significant evidence of a difference in MonthlyCharges between the two churn groups.")
else:
    print("Result: Fail to reject the null hypothesis.")
    print("There is insufficient statistical evidence of a difference in MonthlyCharges between the two churn groups.")

Result: Reject the null hypothesis.
There is statistically significant evidence of a difference in MonthlyCharges between the two churn groups.


In [6]:
monthly_charge_summary = df.groupby("Churn")["MonthlyCharges"].agg(
    ["count", "mean", "median", "std"]
)

monthly_charge_summary.round(2)

,count,mean,median,std
Churn,,,,
No,5174,61.27,64.43,31.09
Yes,1869,74.44,79.65,24.67


## 4. Welch's t-test – Tenure vs Churn

We compare customer tenure between churned and non-churned customers
to determine whether the average tenure differs significantly between
the two groups.

In [7]:
tenure_no_churn = df.loc[
    df["Churn"] == "No",
    "tenure"
]

tenure_churn = df.loc[
    df["Churn"] == "Yes",
    "tenure"
]

t_stat_tenure, p_value_tenure = stats.ttest_ind(
    tenure_no_churn,
    tenure_churn,
    equal_var=False
)

print("t-statistic:", t_stat_tenure)
print("p-value:", p_value_tenure)

t-statistic: 34.823818696312976
p-value: 1.1954945472607151e-232


In [8]:
if p_value_tenure < 0.05:
    print("Statistically significant difference in tenure between churn groups.")
else:
    print("No statistically significant difference detected in tenure.")

Statistically significant difference in tenure between churn groups.


In [9]:
df.groupby("Churn")["tenure"].agg(
    ["count", "mean", "median", "std"]
).round(2)

,count,mean,median,std
Churn,,,,
No,5174,37.57,38.0,24.11
Yes,1869,17.98,10.0,19.53


## 5. Chi-square Test – Contract vs Churn

A chi-square test of independence is used to determine whether
Contract type and Churn are statistically associated.

### Null Hypothesis

Contract type and churn are independent.

### Alternative Hypothesis

Contract type and churn are not independent.

In [10]:
contract_table = pd.crosstab(
    df["Contract"],
    df["Churn"]
)

contract_table

Churn,No,Yes
Contract,,
Month-to-month,2220,1655
One year,1307,166
Two year,1647,48


In [11]:
chi2, p_value_chi, dof, expected = stats.chi2_contingency(
    contract_table
)

print("Chi-square statistic:", chi2)
print("p-value:", p_value_chi)
print("Degrees of freedom:", dof)

Chi-square statistic: 1184.5965720837926
p-value: 5.863038300673391e-258
Degrees of freedom: 2


In [12]:
if p_value_chi < 0.05:
    print("Reject the null hypothesis.")
    print("Contract type and churn are statistically associated.")
else:
    print("Fail to reject the null hypothesis.")
    print("Insufficient evidence of an association between Contract type and churn.")

Reject the null hypothesis.
Contract type and churn are statistically associated.


## 6. Cramér's V – Contract vs Churn

Cramér's V is used to estimate the strength of association between
two categorical variables.

Unlike the chi-square p-value, which addresses statistical significance,
Cramér's V provides an effect-size measure of the association.

In [13]:
n = contract_table.to_numpy().sum()

phi2 = chi2 / n

r, k = contract_table.shape

cramers_v = np.sqrt(
    phi2 / min(k - 1, r - 1)
)

print("Cramér's V:", cramers_v)

Cramér's V: 0.4101156965761409


## 7. Chi-square Test – Internet Service vs Churn

A chi-square test of independence is used to determine whether
InternetService and Churn are statistically associated.

### Null Hypothesis

Internet service type and churn are independent.

### Alternative Hypothesis

Internet service type and churn are not independent.

In [16]:
internet_table = pd.crosstab(
    df["InternetService"],
    df["Churn"]
)

internet_table

Churn,No,Yes
InternetService,,
DSL,1962,459
Fiber optic,1799,1297
No,1413,113


In [17]:
chi2_internet, p_internet, dof_internet, expected_internet = stats.chi2_contingency(
    internet_table
)

print("Chi-square statistic:", chi2_internet)
print("p-value:", p_internet)
print("Degrees of freedom:", dof_internet)

Chi-square statistic: 732.309589667794
p-value: 9.571788222840544e-160
Degrees of freedom: 2


In [18]:
if p_internet < 0.05:
    print("Reject the null hypothesis.")
    print("Internet service type and churn are statistically associated.")
else:
    print("Fail to reject the null hypothesis.")
    print("Insufficient evidence of an association.")

Reject the null hypothesis.
Internet service type and churn are statistically associated.


In [19]:
n_internet = internet_table.to_numpy().sum()

phi2_internet = chi2_internet / n_internet

r_internet, k_internet = internet_table.shape

cramers_v_internet = np.sqrt(
    phi2_internet / min(k_internet - 1, r_internet - 1)
)

print("Cramér's V:", cramers_v_internet)

Cramér's V: 0.32245455521230887


In [20]:
statistical_results = pd.DataFrame({
    "Test": [
        "Monthly Charges vs Churn",
        "Tenure vs Churn",
        "Contract vs Churn",
        "Internet Service vs Churn"
    ],
    "Test_Type": [
        "Welch t-test",
        "Welch t-test",
        "Chi-square",
        "Chi-square"
    ],
    "P_Value": [
        p_value,
        p_value_tenure,
        p_value_chi,
        p_internet
    ]
})

statistical_results["Significant_at_0.05"] = (
    statistical_results["P_Value"] < 0.05
)

statistical_results

,Test,Test_Type,P_Value,Significant_at_0.05
0,Monthly Charges vs Churn,Welch t-test,8.592449e-73,True
1,Tenure vs Churn,Welch t-test,1.195495e-232,True
2,Contract vs Churn,Chi-square,5.863038e-258,True
3,Internet Service vs Churn,Chi-square,9.571788e-160,True


In [21]:
statistical_results.to_csv(
    "../data/processed/statistical_test_results.csv",
    index=False
)

print("Statistical results saved successfully.")

Statistical results saved successfully.
